# 09. External validation: VMBS and PIMCO's mortgage funds

**The question.** Does the frozen recipe behave the same way on mortgage funds it was never developed on, including PIMCO's, and how does an actively managed PIMCO fund's exposure compare with the index ETF's?

**Why a portfolio manager cares.** A result that holds only for one ETF's price series could be an artifact of that fund. VMBS checks fund-specific noise. PIMCO's GNMA fund holds a different mix of mortgages and is actively managed, so it is a harder test, and it answers the question a PIMCO client would ask: does the hedge carry over to PIMCO's own fund?

**Learning objectives**

1. Re-estimate the frozen recipe on VMBS's own history (recipe transfer) and evaluate it on the same test period.
2. Apply MBB's fitted refinancing coefficients to VMBS unchanged (coefficient transfer).
3. Compare MBB and VMBS monthly exposures.
4. Run the same two transfers on PIMCO's GNMA fund (PDMIX).
5. Compare PIMCO's fund with the index: average exposure, and how each moves with refinancing conditions.
6. Describe PMBS's empirical duration since its September 2024 launch, without inferring PIMCO's trades.
7. Run the frozen recipe on PIMCO's Mortgage-Backed Securities Fund before its ETF conversion (PTRIX), if its NAV history is supplied.

| | |
|---|---|
| Inputs | daily and monthly panels, MBB test forecasts, freeze record, optional `data/local_inputs/pimco_ptrix_nav.csv` |
| Outputs | `results/external/*`, figures `09_*` |
| Run after | 07 (and 08 for the MBB comparison numbers) |
| Approximate runtime | about a minute |

> **Two different transfer experiments.** *Recipe transfer* re-runs the same frozen procedure (same half-life, same model forms, same windows) on a fund's own past data, so its coefficients are that fund's own. *Coefficient transfer* keeps MBB's walk-forward refinancing coefficients and applies them to the other fund. VMBS tracks a closely related index, so it checks fund-specific noise and is expected to agree closely with MBB. PIMCO's GNMA fund is a harder test: different holdings and active management.

In [ ]:
# Setup: find the project root, load the configuration, and pick the run mode.
# The mode comes from MBSHEDGE_MODE (set by run_project.py) or config/project.yaml.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config" / "project.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mbshedge.config import setup, experiment
from mbshedge import plotting as pt

cfg, paths, MODE = setup()
EXP = experiment(cfg, MODE)
pt.setup_style()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [ ]:
from mbshedge import pipeline as pl
from mbshedge import evaluation as ev
from mbshedge import exposure as ex
from mbshedge import freeze as fz
from mbshedge.models import static_duration
from mbshedge.sources import parse_local_prices
from mbshedge.prepare import total_return_from_prices, _compound_to_calendar
rec = fz.require_freeze(cfg, paths)
fz.log_test_evaluation(paths, rec, "09", "external validation")
daily = pd.read_parquet(paths.processed / "daily_panel.parquet")
monthly = pd.read_parquet(paths.processed / "monthly_panel.parquet")
fc_test = pl.load_forecasts(paths.results / "forecasts" / "forecasts_test.csv")
choices = rec["development_choices"]; halflife = int(choices["ewma_halflife_days"])
win = pl.windows(daily, EXP); H, EV = EXP["hedging"], EXP["evaluation"]; N = EXP["notional_usd"]
PAIRS = [tuple(p) for p in EXP["models"]["primary_comparisons"]]
xdir = paths.results / "external"; xdir.mkdir(parents=True, exist_ok=True)
status = {}
SYN = " [SYNTHETIC]" if MODE == "smoke" else ""
FUND_COLOR = {"MBB": "#2a78d6", "VMBS": "#eb6834", "PDMIX": "#1baf7a", "PTRIX": "#eda100"}

## 1. VMBS: the frozen recipe on a second fund

In [ ]:
quality_file = paths.results / "data_quality" / "quality_report.json"
quality = json.loads(quality_file.read_text()) if quality_file.exists() else {}

def why_missing(ticker, default):
    """The reason a series is absent from the panel: excluded by the total-return check, or not downloaded."""
    q = quality.get(f"prices_{ticker}", {})
    if q.get("excluded"):
        return {"status": "excluded", "reason": q["excluded"]}
    return {"status": "unavailable", "reason": default}

def recipe_transfer(ret_col, d=None, w=None, models=("S", "U", "P", "AR", "E", "H", "X"), strategies=("UNH", "S", "U", "E", "H", "X"), seed_offset=21):
    """The frozen recipe re-estimated on another fund's own history and evaluated on the test period."""
    d = daily if d is None else d
    w = win if w is None else w
    mon = pl.monthly_panel(d, EXP, [halflife], ret_col=ret_col)
    first = mon[mon["valid"]].index.min()
    static = static_duration(mon, str(first), EXP["sample"]["development_end"])
    fc = pl.forecasts(mon, w, "test", static, halflife, list(models))
    res = pl.run_strategies(d, fc, list(strategies), w["test"]["end_date"], delay=H["base_delay_days"], rule=H["base_rule"],
                            cost_bp=H["base_cost_bp"], financing_bp=H["financing_spread_bp_per_year"], max_h=H["max_hedge_ratio"], ret_col=ret_col)
    evr = pl.evaluate(res, EXP, N, PAIRS, EXP["seed"] + seed_offset)
    return mon, first, static, fc, evr

have_vmbs = EXP["external"]["vmbs_enabled"] and "r_vmbs_ex" in daily and daily["r_vmbs_ex"].notna().sum() > 750
if not have_vmbs:
    status["VMBS"] = why_missing("VMBS", "VMBS prices missing or too short (see results/data_status/access_status.csv)")
    print("VMBS not used: external validation skipped.", status["VMBS"])
else:
    mon_v, first_v, static_v, fc_v, evv = recipe_transfer("r_vmbs_ex", seed_offset=21)
    evv["summary"].to_csv(xdir / "vmbs_recipe_main_table.csv"); evv["bootstrap"].to_csv(xdir / "vmbs_recipe_bootstrap.csv", index=False)
    evv["verdicts"].to_csv(xdir / "vmbs_recipe_verdicts.csv", index=False)
    status["VMBS"] = {"status": "ran", "first_valid_month": str(first_v), "static_duration": static_v, "weeks": int(len(evv["weekly_matrix"]))}
    print(f"VMBS: first valid month {first_v}, static duration {static_v:.2f} years (VMBS development months), {len(evv['weekly_matrix'])} test weeks")
    display(evv["summary"][["vol_gross_ann_pct", "var_reduction_vs_unhedged", "residual_duration_years", "turnover_per_year", "cost_drag_bp_per_year"]].round(3))
    display(evv["bootstrap"][["challenger", "comparator", "variance_reduction", "vr_ci_low", "vr_ci_high", "p_one_sided"]].round(4))

## 2. Coefficient transfer: MBB's refinancing coefficients applied to VMBS

In [ ]:
def coefficient_transfer(mon_t, fc_t, ret_col, seed):
    """Hedge another fund with MBB's walk-forward refinancing coefficients, kept unchanged."""
    coef = fc_test[fc_test["model"] == "E"].set_index("decision_month")[["c0", "c1"]]
    m = mon_t.join(coef, how="inner")
    ct = []
    for dm, r in m.iterrows():
        base_row = fc_t[(fc_t["decision_month"] == dm) & (fc_t["model"] == "U")]
        if base_row.empty:
            continue
        b = base_row.iloc[0].to_dict()
        for name, f in [("E", r["c0"] + r["c1"] * r["g"]), ("H", r[f"U_{halflife}"] + r["c1"] * (r["g"] - r[f"gbar_{halflife}"]))]:
            ct.append({**b, "model": name, "forecast": f})
    fc_ct = pd.concat([fc_t[fc_t["model"].isin(["S", "U"])], pd.DataFrame(ct)], ignore_index=True)
    res_ct = pl.run_strategies(daily, fc_ct, ["S", "U", "E", "H"], win["test"]["end_date"], delay=H["base_delay_days"], rule=H["base_rule"],
                               cost_bp=H["base_cost_bp"], financing_bp=H["financing_spread_bp_per_year"], max_h=H["max_hedge_ratio"], ret_col=ret_col)
    Wct = pd.DataFrame({s: r["weekly"]["gross_pnl"] for s, r in res_ct.items()})
    return ev.paired_bootstrap(Wct, [("H", "U"), ("E", "U")], EV["bootstrap_reps"], EV["bootstrap_block_weeks"], seed)

if have_vmbs:
    bct = coefficient_transfer(mon_v, fc_v, "r_vmbs_ex", EXP["seed"] + 22)
    bct.to_csv(xdir / "vmbs_coefficient_transfer_bootstrap.csv", index=False)
    display(bct[["challenger", "comparator", "variance_reduction", "vr_ci_low", "vr_ci_high"]].round(4))

## 3. Do MBB and VMBS show the same exposure?

In [ ]:
if have_vmbs:
    both = monthly[["D", "se", "valid"]].join(mon_v[["D", "se", "valid"]], rsuffix="_vmbs").loc[win["test_start"]:win["test_end"]]
    both = both[both["valid"] & both["valid_vmbs"]]
    diff = both["D_vmbs"] - both["D"]
    comp = {"months": int(len(both)), "corr": float(both[["D", "D_vmbs"]].corr().iloc[0, 1]), "mean_diff_years": float(diff.mean()),
            "mean_diff_se_years": float(ev.hac_mean_se(diff.to_numpy(), EV["hac_lags_monthly"]))}
    pd.DataFrame([comp]).to_csv(xdir / "exposure_comparison_mbb_vmbs.csv", index=False)
    print(f"Test-period months {comp['months']}: correlation of monthly durations {comp['corr']:.2f}, VMBS minus MBB "
          f"{comp['mean_diff_years']:+.2f} +/- {comp['mean_diff_se_years']:.2f} years (HAC se).")

## 4. PIMCO's GNMA fund (PDMIX): the frozen recipe on PIMCO's own mortgage fund

PDMIX is the Institutional class of the PIMCO GNMA and Government Securities Fund, an actively managed mutual fund benchmarked to the Bloomberg GNMA Index, with daily NAV and distributions since July 1997. Notebook 02 checks its total return against Yahoo's adjusted close, as for the ETFs. It differs from MBB in two ways. It holds mostly Ginnie Mae mortgages (FHA and VA loans, whose borrowers refinance differently from conventional borrowers), and its managers choose coupons, sectors, and duration. The same frozen recipe, costs, and test period are applied, first re-estimated on PDMIX's own past (recipe transfer) and then with MBB's refinancing coefficients (coefficient transfer).

These results describe how the fund's value moved with Treasury yields. They say nothing about PIMCO's trades, hedges, or skill. A mutual fund's NAV is struck from evaluated prices at the 4 pm close, so it has no ETF premium or discount.

In [ ]:
have_pdmix = EXP["external"].get("pdmix_enabled", False) and "r_pdmix_ex" in daily and daily["r_pdmix_ex"].notna().sum() > 750
if not have_pdmix:
    status["PDMIX"] = why_missing("PDMIX", "PDMIX prices missing or too short (see results/data_status/access_status.csv)")
    print("PIMCO GNMA fund not used:", status["PDMIX"])
else:
    mon_g, first_g, static_g, fc_g, evg = recipe_transfer("r_pdmix_ex", seed_offset=24)
    evg["summary"].to_csv(xdir / "pdmix_recipe_main_table.csv"); evg["bootstrap"].to_csv(xdir / "pdmix_recipe_bootstrap.csv", index=False)
    evg["verdicts"].to_csv(xdir / "pdmix_recipe_verdicts.csv", index=False)
    bct_g = coefficient_transfer(mon_g, fc_g, "r_pdmix_ex", EXP["seed"] + 25)
    bct_g.to_csv(xdir / "pdmix_coefficient_transfer_bootstrap.csv", index=False)
    status["PDMIX"] = {"status": "ran", "first_valid_month": str(first_g), "static_duration": static_g, "weeks": int(len(evg["weekly_matrix"]))}
    print(f"PIMCO GNMA fund: first valid month {first_g}, static duration {static_g:.2f} years (development months), {len(evg['weekly_matrix'])} test weeks")
    display(evg["summary"][["vol_gross_ann_pct", "var_reduction_vs_unhedged", "residual_duration_years", "turnover_per_year", "cost_drag_bp_per_year"]].round(3))
    print("Recipe re-estimated on PDMIX:")
    display(evg["bootstrap"][["challenger", "comparator", "variance_reduction", "vr_ci_low", "vr_ci_high", "p_one_sided"]].round(4))
    display(evg["verdicts"].round(4))
    print("MBB's refinancing coefficients applied to PDMIX:")
    display(bct_g[["challenger", "comparator", "variance_reduction", "vr_ci_low", "vr_ci_high"]].round(4))

## 5. PIMCO's fund against the index: exposure and refinancing sensitivity (descriptive)

Month by month, each fund's realized empirical duration is set next to MBB's, and both are related to the share of borrowers in the money at the start of the month (what an investor knew when setting that month's hedge). The slope on that share is the change in duration, in years, from no borrowers in the money to all of them. The difference of two funds' slopes measures how much more, or less, a fund's exposure moves with refinancing conditions than the index's. For PIMCO's GNMA fund it cannot separate the effect of PIMCO's management from the effect of holding Ginnie Mae rather than conventional mortgages. The test-period rows are the declared comparison. The full-sample rows add the development years, when the refinancing share before mid-2013 comes from the retrospective proxy.

In [ ]:
exposure_rows = []
def add_comparison(label, mon_t):
    for sample, a, b in [("test period", win["test_start"], win["test_end"]), ("full sample", win["first_month"], win["test_end"])]:
        c = ex.compare_exposure(mon_t, monthly, a, b, EV["hac_lags_monthly"])
        exposure_rows.append({"fund": label, "sample": sample, **c})

if have_vmbs:
    add_comparison("VMBS", mon_v)
if have_pdmix:
    add_comparison("PDMIX", mon_g)
SHOW_CMP = ["fund", "sample", "months", "corr", "mean_diff_years", "mean_diff_se_years", "slope_x_years", "slope_ref_years", "slope_diff_years", "slope_diff_se"]
cmp_df = pd.DataFrame(exposure_rows)
if len(cmp_df):
    display(cmp_df[[c for c in SHOW_CMP if c in cmp_df]].round(3))

def pimco_vs_index_figure(mon_t, ticker, label):
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3), gridspec_kw={"width_ratios": [2.2, 1]})
    ax = axes[0]
    for lab, mp, col in [("MBB (index ETF)", monthly, FUND_COLOR["MBB"]), (label, mon_t, FUND_COLOR[ticker])]:
        v = mp[mp["valid"]]
        ax.scatter(v["decision_date"], v["D"], s=6, color=col, alpha=0.3)
        ax.plot(mp["decision_date"], mp[f"U_{halflife}"], color=col, lw=1.6, label=f"{lab}, updating estimate")
    ax.axvline(pd.Timestamp(EXP["sample"]["test_start"] + "-01"), color=pt.INK_2, lw=1.0)
    ax.set_ylabel("empirical duration (years)"); ax.set_title(f"{label} and the index ETF (dots: monthly estimates)" + SYN); ax.legend(fontsize=8, loc="lower left")
    ax = axes[1]
    pr = mon_t[["D", "valid"]].join(monthly[["D", "valid"]], lsuffix="_x", rsuffix="_ref")
    pr["g_start"] = monthly["g"].shift(1)
    pr = pr[pr["valid_x"] & pr["valid_ref"] & pr["g_start"].notna()].copy()
    pr["gap"] = pr["D_x"] - pr["D_ref"]
    pr["period"] = np.where(pr.index <= win["development_end"], "development", "test")
    dv, ts = pr[pr["period"] == "development"], pr[pr["period"] == "test"]
    ax.scatter(dv["g_start"], dv["gap"], s=14, facecolors="none", edgecolors=FUND_COLOR[ticker], label="development months")
    ax.scatter(ts["g_start"], ts["gap"], s=14, color=FUND_COLOR[ticker], label="test months")
    row = cmp_df[(cmp_df["fund"] == ticker) & (cmp_df["sample"] == "test period")]
    if len(row) and row.iloc[0].get("status") == "ok":
        r = row.iloc[0]; xs = np.linspace(0, 1, 20)
        ax.plot(xs, r["intercept_diff_years"] + r["slope_diff_years"] * xs, color=pt.INK, lw=1.2,
                label=f"test-period fit, slope {r['slope_diff_years']:+.2f} (95% CI {r['slope_diff_ci_low']:+.2f} to {r['slope_diff_ci_high']:+.2f})")
    ax.axhline(0, color=pt.INK_2, lw=0.8)
    ax.set_xlabel("share of borrowers in the money, start of month"); ax.set_ylabel(f"{ticker} minus MBB duration (years)")
    ax.set_title("Difference against refinancing conditions"); ax.legend(fontsize=7.5, scatterpoints=1)
    pt.save_figure(fig, f"09_{ticker.lower()}_vs_index", paths, data=pr, mode=MODE)
    plt.show()

if have_pdmix:
    pimco_vs_index_figure(mon_g, "PDMIX", "PIMCO GNMA fund (PDMIX)")

## 6. PMBS (PIMCO Mortgage-Backed Securities Active ETF), descriptive

PMBS has traded since September 23, 2024, about two years in a single locked-in regime, so no hedge test is run on it. Below are its monthly empirical durations next to MBB's. Differences reflect an active portfolio (coupon and sector choices, derivatives, possibly non-agency holdings) and cannot identify PIMCO's trades or intentions. ETF market prices can also differ from NAV.

In [ ]:
have_pmbs = EXP["external"]["pmbs_descriptive_enabled"] and "r_pmbs_ex" in daily and daily["r_pmbs_ex"].notna().sum() > 120
if not have_pmbs:
    status["PMBS"] = why_missing("PMBS", "PMBS prices missing or too short")
    print("PMBS not used.", status["PMBS"])
else:
    mon_p = pl.monthly_panel(daily, EXP, [halflife], ret_col="r_pmbs_ex")
    first_full = (daily["r_pmbs_ex"].first_valid_index().to_period("M") + 1)
    pm = mon_p.loc[first_full:win["test_end"], ["decision_date", "D", "se", "valid", f"U_{halflife}"]].join(
        monthly[["D", "se", "valid", f"U_{halflife}"]], rsuffix="_mbb")
    pm = pm[pm["valid"] & pm["valid_mbb"]]
    pm.to_csv(xdir / "pmbs_descriptive.csv")
    d = pm["D"] - pm["D_mbb"]
    status["PMBS"] = {"status": "descriptive only", "months": int(len(pm)), "mean_diff_years": float(d.mean()),
                      "mean_diff_se_years": float(ev.hac_mean_se(d.to_numpy(), 2)) if len(pm) > 6 else None,
                      "latest_updating_pmbs": float(pm[f"U_{halflife}"].iloc[-1]) if len(pm) else None,
                      "latest_updating_mbb": float(pm[f"U_{halflife}_mbb"].iloc[-1]) if len(pm) else None}
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.errorbar(pm["decision_date"] - pd.Timedelta(days=3), pm["D_mbb"], yerr=pm["se_mbb"], fmt="o", ms=4, color="#2a78d6", ecolor="#cde2fb", label="MBB (index ETF)")
    ax.errorbar(pm["decision_date"] + pd.Timedelta(days=3), pm["D"], yerr=pm["se"], fmt="s", ms=4, color="#eb6834", ecolor="#fbd3c2", label="PMBS (PIMCO active ETF)")
    ax.set_ylabel("years"); ax.set_title("Monthly empirical duration since PMBS launched (descriptive)"); ax.legend(fontsize=8.5)
    pt.save_figure(fig, "09_pmbs_vs_mbb", paths, data=pm, mode=MODE)
    plt.show()
    print(json.dumps(status["PMBS"], indent=1))

## 7. PIMCO Mortgage-Backed Securities Fund before the ETF conversion (PTRIX)

PIMCO converted this fund into the PMBS ETF in September 2024. On 2026-11-02 Yahoo returned no observations for PTRIX or PMRPX (both symbols now map to the PMBS ETF). The section runs only if you place the predecessor fund's **daily NAV and distributions** in `data/local_inputs/pimco_ptrix_nav.csv` with columns `date, close, dividend` (NAV in `close`, cash distribution on its ex-date in `dividend`). `tools/convert_pimco_nav.py` builds that file from a CRSP, Bloomberg, or PIMCO export. It then reruns the frozen recipe on the overlap of that history with the test period, and adds the fund to the exposure comparison. NAV-based and market-price series must not be spliced without an adjustment, so this route uses the NAV history alone.

In [ ]:
pimco_file = paths.local_inputs / cfg["data"]["pimco_local_file"]
have_ptrix = False
if not pimco_file.exists():
    status["PIMCO_predecessor"] = {"status": "unavailable",
                                   "reason": "No user-supplied NAV history. Verified 2026-11-02: Yahoo has no PTRIX/PMRPX history after the 2024 ETF conversion.",
                                   "how_to_enable": f"save daily NAV and distributions as {paths.rel(pimco_file)} (columns date, close, dividend), for example with tools/convert_pimco_nav.py"}
    print(json.dumps(status["PIMCO_predecessor"], indent=1))
else:
    nav = parse_local_prices(pimco_file.read_bytes())
    tr, q = total_return_from_prices(nav)
    d2 = daily.copy()
    d2["tr_pimco"] = _compound_to_calendar(tr, d2.index)
    d2["r_pimco_ex"] = d2["tr_pimco"] - d2["rf"]
    mon_x = pl.monthly_panel(d2, EXP, [halflife], ret_col="r_pimco_ex")
    last_x = mon_x[mon_x["valid"]].index.max()
    w2 = {**win, "test": {**win["test"], "last_decision": min(win["test"]["last_decision"], last_x - 1),
                          "end_date": d2.index[d2.index.to_period("M") == min(win["test_end"], last_x)].max()}}
    mon_x, first_x, static_x, fc_x, evx = recipe_transfer("r_pimco_ex", d=d2, w=w2, models=("S", "U", "E", "H"), strategies=("UNH", "S", "U", "E", "H"), seed_offset=23)
    evx["bootstrap"].to_csv(xdir / "pimco_predecessor_bootstrap.csv", index=False)
    evx["summary"].to_csv(xdir / "pimco_predecessor_main_table.csv"); evx["verdicts"].to_csv(xdir / "pimco_predecessor_verdicts.csv", index=False)
    status["PIMCO_predecessor"] = {"status": "ran on user-supplied NAV", "file": paths.rel(pimco_file), "first_valid_month": str(first_x),
                                   "last_valid_month": str(last_x), "test_weeks": int(len(evx["weekly_matrix"])), "quality": q}
    have_ptrix = True
    display(evx["bootstrap"][["challenger", "comparator", "variance_reduction", "vr_ci_low", "vr_ci_high", "p_one_sided"]].round(4))
    add_comparison("PTRIX", mon_x)
    cmp_df = pd.DataFrame(exposure_rows)
    display(cmp_df[cmp_df["fund"] == "PTRIX"][[c for c in SHOW_CMP if c in cmp_df]].round(3))
    pimco_vs_index_figure(mon_x, "PTRIX", "PIMCO MBS fund (PTRIX)")
cmp_df.to_csv(xdir / "exposure_comparison_vs_mbb.csv", index=False)
(xdir / "external_status.json").write_text(json.dumps(status, indent=2, default=str))

## 8. The external results next to MBB

In [ ]:
mbb_boot = pd.read_csv(paths.results / "evaluation" / "bootstrap_test.csv") if (paths.results / "evaluation" / "bootstrap_test.csv").exists() else None
rows = []
def collect(asset, fund, method, b):
    for _, r in b.iterrows():
        rows.append({"asset": asset, "fund": fund, "method": method, "comparison": f"{r['challenger']} vs {r['comparator']}",
                     "vr": r["variance_reduction"], "lo": r["vr_ci_low"], "hi": r["vr_ci_high"]})
if mbb_boot is not None:
    collect("MBB (primary)", "MBB", "primary", mbb_boot[mbb_boot["primary"]])
if have_vmbs:
    collect("VMBS, own recipe", "VMBS", "recipe", evv["bootstrap"])
    collect("VMBS, MBB coefficients", "VMBS", "coefficients", bct)
if have_pdmix:
    collect("PIMCO GNMA fund, own recipe", "PDMIX", "recipe", evg["bootstrap"])
    collect("PIMCO GNMA fund, MBB coefficients", "PDMIX", "coefficients", bct_g)
if have_ptrix:
    collect("PIMCO MBS fund (PTRIX), own recipe", "PTRIX", "recipe", evx["bootstrap"])
ext = pd.DataFrame(rows); ext.to_csv(xdir / "external_comparison.csv", index=False)
if len(ext):
    comps = ["H vs U", "E vs U", "U vs S", "H vs S"]
    assets = list(dict.fromkeys(ext["asset"]))
    k = len(assets); step = 0.8 / max(k, 1)
    marker = {"primary": "o", "recipe": "o", "coefficients": "s"}
    fig, ax = plt.subplots(figsize=(9.5, 1.2 + 0.9 * len(comps) + 0.12 * k))
    for j, a in enumerate(assets):
        d = ext[ext["asset"] == a]
        col = FUND_COLOR.get(d["fund"].iloc[0], pt.INK); mk = marker[d["method"].iloc[0]]
        face = "white" if d["method"].iloc[0] == "coefficients" else col
        for _, r in d.iterrows():
            if r["comparison"] not in comps:
                continue
            y = comps.index(r["comparison"]) + (j - (k - 1) / 2) * step
            ax.errorbar(r["vr"], y, xerr=[[r["vr"] - r["lo"]], [r["hi"] - r["vr"]]], fmt=mk, color=col, mfc=face, ms=5, capsize=0, lw=1.4)
        ax.plot([], [], marker=mk, ls="none", color=col, mfc=face, label=a)
    ax.axvline(0, color=pt.INK_2, lw=0.8)
    ax.set_yticks(range(len(comps))); ax.set_yticklabels(comps); ax.invert_yaxis()
    ax.set_xlabel("test-period variance reduction (95% paired bootstrap interval)"); ax.set_title("Primary comparisons on MBB, VMBS, and PIMCO's funds" + SYN)
    ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
    pt.save_figure(fig, "09_external_validation", paths, data=ext, mode=MODE)
    plt.show()